In [15]:
%%time
%matplotlib inline
import rioxarray
import numpy as np
import importlib 
import matplotlib.pyplot as plt
import geopandas as gpd

CPU times: total: 0 ns
Wall time: 0 ns


In [16]:
ndvi_data = rioxarray.open_rasterio('../input/STNDVI_202223.tif')
vh_data = rioxarray.open_rasterio('../input/STVH_202223.tif')
import rioxarray

input_path = "../input/ST_Song.tif"
mask_data = rioxarray.open_rasterio(input_path)
mask_data

<xarray.DataArray (band: 1, y: 7658, x: 7489)> Size: 57MB
[57350762 values with dtype=uint8]
Coordinates:
  * band         (band) int64 8B 1
  * x            (x) float64 60kB 5.597e+05 5.597e+05 ... 6.346e+05 6.346e+05
  * y            (y) float64 61kB 1.099e+06 1.099e+06 ... 1.022e+06 1.022e+06
    spatial_ref  int64 8B 0
Attributes:
    AREA_OR_POINT:       Area
    RepresentationType:  THEMATIC
    _FillValue:          255
    scale_factor:        1.0
    add_offset:          0.0

In [17]:
import xarray as xr
import numpy as np

# Assuming 'data' is your xarray.DataArray
mask = mask_data != 255  # Boolean mask where values are not 255

# Stack x and y dimensions into a single index
data_stacked = mask_data.stack(points=("y", "x"))

# Apply the mask and drop NaN values
filtered = data_stacked.where(data_stacked != 255, drop=True)

# Extract x and y coordinates
x_indices = filtered.coords["x"].values
y_indices = filtered.coords["y"].values

# Stack into a (N, 2) NumPy array
xy_valid = np.column_stack((x_indices, y_indices))

print(xy_valid)  # Print or use the filtered coordinates

[[ 598452.754  1098615.3978]
 [ 598442.754  1098605.3978]
 [ 598452.754  1098605.3978]
 ...
 [ 591022.754  1022045.3978]
 [ 591032.754  1022045.3978]
 [ 591042.754  1022045.3978]]


In [18]:
i = 0
ndvi_list = []
vh_list = []
for index in xy_valid:
    data = ndvi_data.sel(x=index[0], y=index[1], method='nearest').values
    vh = vh_data.sel(x=index[0], y=index[1], method='nearest').values
    if not np.any(np.isnan(data)):
        ndvi_list.append(data)
        vh_list.append(vh)

In [19]:
len(ndvi_list)

390

In [20]:

# Lọc các mảng không chứa nan
filtered_arrays = [arr for arr in ndvi_list if not np.any(np.isnan(arr))]

# In kết quả
if filtered_arrays:
    for arr in filtered_arrays:
        print(arr)
else:
    print("Không có mảng nào trong dữ liệu thỏa mãn điều kiện (không chứa nan).")

[ 0.5855629   0.81481487 -0.05464005  0.7717799   0.73463273  0.77937776
  0.7421972   0.72383344  0.69376135  0.8152413   0.8149537   0.797909
  0.7227723 ]
[ 0.58188826  0.84111226 -0.05892591  0.8077283   0.7281624   0.7929113
  0.7471309   0.7546101   0.72406214  0.8415024   0.83339673  0.79444945
  0.74514943]
[ 0.6750902   0.6958358  -0.05417276  0.78047395  0.70264906  0.6974375
  0.71602625  0.8249132   0.54561853  0.7205495   0.79972816  0.7773169
  0.45127615]
[ 0.71101916  0.7733473  -0.06377929  0.8111117   0.7198986   0.74105626
  0.7408115   0.8505523   0.6210847   0.79587287  0.83761734  0.80214834
  0.5565313 ]
[ 0.59851307  0.4865293  -0.05896296  0.7672397   0.5689655   0.6178011
  0.7202234   0.8238045   0.20662658  0.7377245   0.8335673   0.7749164
  0.20948108]
[ 0.62568605  0.55095786 -0.06570224  0.7781739   0.56700313  0.62856084
  0.7129597   0.8200569   0.23573281  0.8055948   0.8594996   0.7836615
  0.22857141]
[-0.01953418 -0.02550258  0.00821917 -0.01666754

In [ ]:

np.save('../input/nonan_song_ndvi.npy',ndvi_list)
np.save('../input/nonan_song_vh.npy',vh_list)
len(vh_list)


390